# Module 5 · Worksheet 2
# Recursive Partitioning, Gini, and Entropy

**ISA 591 — Introduction to Data Mining in Business**  
**Focus:** Node impurity, weighted split quality, greedy splitting

## Learning goals

1. Calculate Gini impurity for binary and multiclass nodes.
2. Calculate entropy and distinguish it from information gain.
3. Weight child impurities correctly and compare candidate splits.
4. Connect hand calculations to the values stored in a fitted tree.
5. Explain the role of class imbalance and training-only resampling.

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

SEED = 591
pd.set_option('display.max_columns', 100)
pd.set_option('display.precision', 4)
print('scikit-learn:', sklearn.__version__)

scikit-learn: 1.5.2


## 1. What does impurity measure?

A **pure** classification node contains one class. A mixed node contains several classes. The tree searches for a split that reduces impurity, then repeats the search within the resulting nodes: **recursive partitioning**.

For class proportions $p_1,\ldots,p_K$,

$$G=1-\sum_{k=1}^{K}p_k^2.$$

For two classes, $G=2p(1-p)$. Binary Gini is largest at a 50/50 mixture, where it equals 0.5. With $K$ equally represented classes, its maximum is $1-1/K$.

### Worked example
A node has 8 customers who stay and 2 who churn:

$$G=1-(8/10)^2-(2/10)^2=0.32.$$

In [1]:
## Example: Review is Class 1: Neg, Class 2: Neut, Class 3: Pos

#### NODE:  [20, 50, 30]
###### y_hat = Neut
###### p_hat = [0.20, 0.50, 0.30]

#### CALCULATE GINI (IMPURITY)
Gini = 1 - (0.2**2 + 0.5**2 + 0.3**2)
print(f'The Gini of the NODE is: {Gini}')

The Gini of the NODE is: 0.62


In [2]:
p1 = 0.01
p2 = 0.98
p3 = 0.01

Gini = 1 - (p1**2+p2**2 +p3**2)
Gini

0.0394000000000001

In [4]:
## BINARY CLASSIFICATION (p1 and p2)

#### Root Node: Non-Frauds: 12, Frauds: 12
p1 = 12/24
p2 = 12/24
Gini = 2*p1*p2
print(f'The Gini for Binary Classification: {Gini}')

1 - (p1**2 + p2**2) == 2*p1*p2 ## the formulas simplify for binary case

The Gini for Binary Classification: 0.5


True

In [5]:
## Example: Class 1: 5, Class 2: 11
p1 = 5/16
p2 = 11/16

Gini = 2*p1*p2
print(f'The Gini of the Right Node is {Gini}')

The Gini of the Right Node is 0.4296875


In [6]:
## POLL QUIZ QUESTION #1
p1 = 9 / 12
p2 = 3 / 12
2*p1*p2

0.375

In [7]:
## POLL QUIZ QUESTION # 2
p1 = 2/5
p2 = 3/5
2*p1*p2

0.48

In [8]:
## POLL QUIZ QUESTION # 3
p1 = 4/12
p2 = 4/12
p3 = 4/12

1 - (p1**2+p2**2+p3**2)

0.6666666666666667

In [ ]:
## is the GINI index on the ROOT NODE always 0.5?
#### No, unless it is a perfectly balanced problem OR we use SMOTE 

In [9]:
## POLL QUIZ QUESTION # 4
(2841/3537)*0.478 + (696/3537)*0.02

0.3878761662425785

In [ ]:
## What is the improvement?
#### The improvement (sometimes called GAIN - MIN IMPURITY DECREASE) is the difference between the GINI on the TOP NODE and the COMBINED GINI of the children nodes

## Is there a cutoff on what improvement is GOOD or BAD?
#### nO, depends on the problem 

In [11]:
## POLL QUIZ QUESTION # 5
gini_root = 0.5
combined_gini = (2841/3537)*.478 + (696/3537)*0.02

impurity_decrease = gini_root - combined_gini
impurity_decrease

0.11212383375742152

### Practice 1A — Binary and multiclass nodes

Calculate Gini for counts `[18, 6]`, `[12, 12]`, `[24, 0]`, and `[4, 4, 4]`. Which binary node is most impure?

In [12]:
# TODO
gini_18_6 = 2*(18/24)*(6/24)
gini_12_12 = 2*(12/24)*(12/24)
gini_pure = 2*(24/24)*(0/24)
gini_three_classes = 1 - ((4/12)**2 + (4/12)**2 +(4/12)**2)

## 2. A split has TWO child nodes

Consider this small **constructed** example. The counts are ordered `[class 0, class 1]`.

| Node | Class 0 | Class 1 | Total |
|---|---:|---:|---:|
| Parent | 12 | 12 | 24 |
| Left child | 7 | 1 | 8 |
| Right child | 5 | 11 | 16 |

Weight each child's impurity by its share of the parent:

$$I_{children}=\frac{n_L}{n}I_L+\frac{n_R}{n}I_R,$$
$$\Delta I=I_{parent}-I_{children}.$$

A smaller weighted child impurity—or equivalently a larger decrease—is better for this local split. The weights sum to 1.

In [13]:
gini_top = 0.5
gini_left = 2*(7/8)*(1/8)
gini_right = 2*(5/16)*(11/16)
combined_gini = (8/24)*gini_left + (16/24)*gini_right

improvement = gini_top - combined_gini
improvement

0.140625

### Practice 2A — Weighted Gini and improvement

Compute each node’s Gini, the combined child Gini, and the improvement. Keep unrounded values until the final answer.

In [ ]:
# TODO
g_parent = None
g_left = None
g_right = None
g_children = None
g_decrease = None

**Interpret / discuss:** Why is `(left impurity + right impurity) / 2` not generally correct? When would that simple average be correct?

**Your response:**

_Write your answer here._

## 3. Entropy and information gain

$$H=-\sum_{k=1}^{K}p_k\log_2(p_k).$$

Use the convention $0\log_2(0)=0$. A pure node has entropy 0. A balanced binary node has entropy 1.

**Terminology note:** entropy measures the impurity of a node. **Information gain** is the reduction in entropy from a split; they are not interchangeable quantities.

In [20]:
### Example: Class 1: 12  Class 2: 12

p1 = 12/24
p2 = 12/24

entropy = -(p1*np.log2(p1) + p2*np.log2(p2))
entropy

1.0

In [21]:
from numpy import log2
p1 = 12/19
p2 = 7/19

-(p1*log2(p1) + p2*log2(p2))

0.9494520153879484

In [ ]:
## NO NEED TO CALCULATE B/C IMPURITY = 0
p1 = 0/5
p2 = 5/5

-(p1*log2(p1) +p2*log2(p2))

C:\Users\martinwg\AppData\Local\Temp\ipykernel_2016\4245454278.py:4: RuntimeWarning: divide by zero encountered in log2
  -(p1*log2(p1) +p2*log2(p2))
C:\Users\martinwg\AppData\Local\Temp\ipykernel_2016\4245454278.py:4: RuntimeWarning: invalid value encountered in scalar multiply
  -(p1*log2(p1) +p2*log2(p2))


nan

### Practice 3A — Entropy improvement

For the same parent and children in Practice 2A, calculate the weighted child entropy and information gain.

In [ ]:
# TODO
h_parent = None
h_children = None
information_gain = None
print('Parent entropy:', h_parent)
print('Weighted child entropy:', h_children)
print('Information gain:', information_gain)

### Run: how the criteria respond to class mixtures

The two curves have different numerical scales. Compare **candidate splits using the same criterion**, not a Gini value directly against an entropy value.

In [ ]:
probabilities = np.linspace(0, 1, 201)
plt.figure(figsize=(8, 4))
plt.plot(probabilities, [gini([1-p, p]) for p in probabilities], label='Gini')
plt.plot(probabilities, [entropy([1-p, p]) for p in probabilities], label='Entropy')
plt.xlabel('Proportion in class 1'); plt.ylabel('Node impurity')
plt.title('Binary-node impurity'); plt.legend(); plt.show()

## 4. Compare two candidate splits

Both splits below partition the same parent `[12, 12]`.

| Candidate | Left counts | Right counts |
|---|---|---|
| A | `[7, 1]` | `[5, 11]` |
| B | `[12, 7]` | `[0, 5]` |

One perfectly pure child is not enough to decide which split is better; inspect the other child and both sample sizes.

In [ ]:
def gini(counts):
    counts = np.asarray(counts, dtype=float)
    if np.any(counts < 0) or counts.sum() <= 0:
        raise ValueError('Counts must be nonnegative with a positive total.')
    p = counts / counts.sum()
    return float(1 - np.sum(p**2))

def entropy(counts):
    counts = np.asarray(counts, dtype=float)
    if np.any(counts < 0) or counts.sum() <= 0:
        raise ValueError('Counts must be nonnegative with a positive total.')
    p = counts / counts.sum()
    p = p[p > 0]  # Avoid log2(0); zero-probability terms contribute zero.
    return float(-np.sum(p * np.log2(p)))

In [ ]:
def split_summary(left, right, criterion):
    left = np.asarray(left)
    right = np.asarray(right)
    parent = left + right
    n = parent.sum()
    combined = (left.sum()/n)*criterion(left) + (right.sum()/n)*criterion(right)
    return criterion(parent), combined, criterion(parent)-combined

rows = []
for candidate, left, right in [('A', [7, 1], [5, 11]), ('B', [12, 7], [0, 5])]:
    for name, function in [('Gini', gini), ('Entropy', entropy)]:
        parent, combined, gain = split_summary(left, right, function)
        rows.append({'Candidate': candidate, 'Criterion': name,
                     'Parent': parent, 'Children': combined, 'Improvement': gain})
split_results = pd.DataFrame(rows)
display(split_results)

**Interpret / discuss:** Which candidate is selected by Gini? Which is selected by entropy? Does this example establish that either criterion is always better on unseen customers?

**Your response:**

_Write your answer here._

## 5. Connect the arithmetic to fitted trees

## Business setting: customer retention

A telecommunications company wants to identify customers who are likely to leave. Each row is one customer. **`Churn = Yes` is the positive class**, encoded as 1; `No` is encoded as 0.

| Variables | Meaning / role |
|---|---|
| `customerID` | Customer identifier; excluded from the predictor matrix. |
| `tenure` | Number of months the customer has stayed with the company. |
| `MonthlyCharges`, `TotalCharges` | Monthly charge and accumulated charges. `TotalCharges` may initially contain nonnumeric entries. |
| `Contract` | Month-to-month, one-year, or two-year contract. |
| `InternetService`, `PhoneService`, `MultipleLines` | Service types. |
| `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | Service options; some levels describe the absence of the underlying service. |
| `PaymentMethod`, `PaperlessBilling` | Payment and billing categories. |
| `gender`, `SeniorCitizen`, `Partner`, `Dependents` | Demographic and household attributes. Numeric storage does not automatically mean a continuous measurement. |
| `Churn` | Outcome, not a predictor. |

This is a classroom historical snapshot. Before deployment, confirm that every input is available at the actual prediction time and assess whether sensitive attributes are appropriate for the intended use.

### Run: load and split the data

The main application throughout this module uses the same Telco customer-churn data. **Fitting data** grow the exploratory models; **validation data** support comparisons and tuning; the **final test data** are held aside until the stated evaluation checkpoint. Together, fitting and validation form the 80% development set.


In [ ]:
## READ DATA
df = pd.read_csv('https://raw.githubusercontent.com/martinwg/ISA591/refs/heads/main/data/customer_churn.csv')

## DROP customerId
df.drop(columns=['CustomerId', 'Lastname'], inplace = True)

## ENCODE CATEGORICAL PREDICTORS
df = pd.get_dummies(df, drop_first = True, dtype = int)

## Create X and y
X = df.drop("Churned", axis = 1)
y = df.Churned

## SPLIT INTO TRAINING (0.8) AND TEST, RANDOM STATE = 591
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 591)

## tree module from sklearn
from sklearn.tree import DecisionTreeClassifier  ## y = {0,1}

## instance (we call max_depth = 3, min_samples_leaf=25 ----  hyperparameters)
## max_depth = 3, min_samples_leaf=25 are STOPPING RULES
## criterion is just the IMPURITY METRIC to minimize
dt = DecisionTreeClassifier(max_depth = 3, min_samples_leaf=25, random_state=591, criterion = "gini")

## fit
dt.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, mean_squared_error, mean_absolute_error
)

def positive_probability(model, frame):
    class_one_column = list(model.classes_).index(1)
    return model.predict_proba(frame)[:, class_one_column]

def classification_metrics(model, frame, truth):
    predicted = model.predict(frame)
    probabilities = positive_probability(model, frame)
    tn, fp, fn, tp = confusion_matrix(truth, predicted, labels=[0, 1]).ravel()
    return {
        'Accuracy': accuracy_score(truth, predicted),
        'Recall': recall_score(truth, predicted, zero_division=0),
        'Specificity': tn / (tn + fp),
        'Precision': precision_score(truth, predicted, zero_division=0),
        'F1': f1_score(truth, predicted, zero_division=0),
        'ROC AUC': roc_auc_score(truth, probabilities)
    }

def targeting_metrics(truth, probabilities, fraction=0.20):
    """Select exactly the top k; ties retain the existing row order."""
    truth = np.asarray(truth)
    probabilities = np.asarray(probabilities)
    k = max(1, int(np.ceil(fraction * len(truth))))
    selected = np.argsort(-probabilities, kind='stable')[:k]
    rate = truth[selected].mean()
    base_rate = truth.mean()
    return {
        'Selected': k,
        'Churners captured': int(truth[selected].sum()),
        'Selected churn rate': rate,
        'Capture rate': truth[selected].sum() / truth.sum(),
        'Lift': rate / base_rate
    }

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
stumps = {}
for criterion in ['gini', 'entropy']:
    model = make_model(DecisionTreeClassifier(
        criterion=criterion, max_depth=1, random_state=SEED
    ))
    model.fit(X_fit, y_fit)
    stumps[criterion] = model
    plt.figure(figsize=(10, 5))
    plot_tree(model.named_steps['model'],
              feature_names=model.named_steps['prep'].get_feature_names_out(),
              class_names=['Stayed', 'Churned'], filled=False, precision=4)
    plt.title(f'Telco churn stump: {criterion}')
    plt.show()

## 6. Optional experiment — Search the thresholds of one predictor

A numeric predictor is split at a threshold. This illustration evaluates candidate midpoints for **tenure only**. A full tree also evaluates eligible candidates for other predictors, then repeats the search within the children. This is a greedy local procedure, not an exhaustive search over every possible complete tree.

In [ ]:
values = np.sort(X_fit['tenure'].unique())
thresholds = (values[:-1] + values[1:]) / 2
threshold_rows = []
for threshold in thresholds:
    in_left = X_fit['tenure'] <= threshold
    left_counts = np.bincount(y_fit.loc[in_left], minlength=2)
    right_counts = np.bincount(y_fit.loc[~in_left], minlength=2)
    parent, combined, gain = split_summary(left_counts, right_counts, gini)
    threshold_rows.append({'Threshold': threshold, 'Weighted Gini': combined,
                           'Improvement': gain})
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.sort_values('Improvement', ascending=False).head())
plt.figure(figsize=(8, 4))
plt.plot(threshold_table['Threshold'], threshold_table['Improvement'], marker='.')
plt.xlabel('Tenure threshold'); plt.ylabel('Gini improvement')
plt.title('Candidate root splits using tenure only'); plt.show()

## Exit ticket

Explain why the following statement is incomplete: “Choose the split with the purest left child.” Include sample sizes, both children, and the distinction between fitting quality and validation performance.

**Your response:**

_Write your answer here._

## Implementation references

These official references support the API details and the explicitly labeled implementation notes. The questions and small constructed examples are classroom exercises.

- [Tree impurity criteria and mathematical formulation](https://scikit-learn.org/stable/modules/tree.html#mathematical-formulation)
- [DecisionTreeRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
- [SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html)
- [SMOTENC for mixed feature types](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTENC.html)
- [Resampling and data leakage](https://imbalanced-learn.org/stable/common_pitfalls.html)